# 01 Preprocessing: UK Biobank phenotype table

Reads the raw UK Biobank extracts from `DATA_ROOT` (default `/data/UKB`), recodes baseline covariates, derives the first-record age of every ICD-10 category in the vocabulary, applies the genetic quality-control filters and the Townsend / NMR completeness filter, and writes `phenotypes.tsv` to `PREPROCESSED_DIR` (default `/preprocessed/UKB`).

**Confidentiality:** the output is participant-level data. Keep it outside the repository and never commit notebook outputs (see the README).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np

import glob
import os

from preprocessing_helpers import *
from paths import DATA_ROOT, PREPROCESSED_DIR

In [ ]:
data_root = DATA_ROOT   # default /data/UKB, override with the UKB_DATA_ROOT environment variable

age = pd.read_csv(os.path.join(data_root, "phenotypes", "Age.txt.gz"), sep="\t")
cf = pd.read_csv(os.path.join(data_root, "phenotypes", "CF.txt.gz"), sep="\t")
other = pd.read_csv(os.path.join(data_root, "phenotypes", "Other.txt.gz"), sep="\t")
adversities = pd.read_csv(os.path.join(data_root, "phenotypes", "Adversities.txt.gz"), sep="\t")

nmr_1 = pd.read_csv(os.path.join(data_root, "phenotypes", "NMR_1.txt.gz"), sep="\t")
nmr_2 = pd.read_csv(os.path.join(data_root, "phenotypes", "NMR_2.txt.gz"), sep="\t")
nmr_3 = pd.read_csv(os.path.join(data_root, "phenotypes", "NMR_3.txt.gz"), sep="\t")
nmr_4 = pd.read_csv(os.path.join(data_root, "phenotypes", "NMR_4.txt.gz"), sep="\t")
nmr_5 = pd.read_csv(os.path.join(data_root, "phenotypes", "NMR_5.txt.gz"), sep="\t")

pa = pd.read_csv(os.path.join(data_root, "phenotypes", "PA.txt.gz"), sep="\t")

diseases_1 = pd.read_csv(os.path.join(data_root, "phenotypes", "Diseases_1.txt.gz"), sep="\t")
diseases_2 = pd.read_csv(os.path.join(data_root, "phenotypes", "Diseases_2.txt.gz"), sep="\t")
diseases_3 = pd.read_csv(os.path.join(data_root, "phenotypes", "Diseases_3.txt.gz"), sep="\t")
diseases_4 = pd.read_csv(os.path.join(data_root, "phenotypes", "Diseases_4.txt.gz"), sep="\t")

other_diseases = pd.read_csv(os.path.join(data_root, "phenotypes", "Other_Diseases.txt.gz"), sep="\t")

In [ ]:
# read ukb_showcase.xlsx to get the column descriptions
showcase = pd.read_excel(os.path.join(data_root, "ukb_showcase.xlsx"), sheet_name=None, engine='openpyxl')

In [ ]:
codings_csv = glob.glob(os.path.join(data_root, "*.codings.csv"))[0]
codings_df = pd.read_csv(codings_csv)

data_dict_csv = glob.glob(os.path.join(data_root, "*.data_dictionary.csv"))[0]
data_dict_df = pd.read_csv(data_dict_csv, low_memory=False)

# Construction of the phenotype dataset

In [ ]:
# remove all columns fully empty
age = age.dropna( axis = 1, how = 'all' )
cf = cf.dropna( axis = 1, how = 'all' )
other = other.dropna( axis = 1, how = 'all' )
adversities = adversities.dropna( axis = 1, how = 'all' )
nmr_1 = nmr_1.dropna( axis = 1, how = 'all' )
nmr_2 = nmr_2.dropna( axis = 1, how = 'all' )
nmr_3 = nmr_3.dropna( axis = 1, how = 'all' )
nmr_4 = nmr_4.dropna( axis = 1, how = 'all' )
nmr_5 = nmr_5.dropna( axis = 1, how = 'all' )
pa = pa.dropna( axis = 1, how = 'all' )
diseases_1 = diseases_1.dropna( axis = 1, how = 'all' )
diseases_2 = diseases_2.dropna( axis = 1, how = 'all' )
diseases_3 = diseases_3.dropna( axis = 1, how = 'all' )
diseases_4 = diseases_4.dropna( axis = 1, how = 'all' )
other_diseases = other_diseases.dropna( axis = 1, how = 'all' )

# merge on eid without suffixes
merged = pd.merge( cf, other, on = 'eid', how = 'inner' )
merged = pd.merge( merged, adversities, on = 'eid', how = 'inner' )
merged = pd.merge( merged, nmr_1, on = 'eid', how = 'inner' )
merged = pd.merge( merged, nmr_2, on = 'eid', how = 'inner' )
merged = pd.merge( merged, nmr_3, on = 'eid', how = 'inner' )
merged = pd.merge( merged, nmr_4, on = 'eid', how = 'inner' )
merged = pd.merge( merged, nmr_5, on = 'eid', how = 'inner' )
merged = pd.merge( merged, pa, on = 'eid', how = 'inner' )
merged = pd.merge( merged, diseases_1, on = 'eid', how = 'inner' )
merged = pd.merge( merged, diseases_2, on = 'eid', how = 'inner' )
merged = pd.merge( merged, diseases_3, on = 'eid', how = 'inner' )
merged = pd.merge( merged, diseases_4, on = 'eid', how = 'inner' )
merged = pd.merge( merged, age, on = 'eid', how = 'inner' )
merged = pd.merge( merged, other_diseases, on = 'eid', how = 'inner' )
# display the first few rows of the merged dataframe
merged.head()

print( f"Merged shape: {merged.shape}" )

In [ ]:
merged.rename( columns = {
    'p21003_i0': 'age_at_assessment',
    'p21003_i2': 'age_at_imaging',
    'p31': 'sex',
    'p22189': 'townsend_deprivation_index',
    'p23099_i0': 'body_fat_percentage_at_assessment',
    'p23099_i2': 'body_fat_percentage_at_imaging',
    'p23104_i0': 'bmi_at_assessment',
    'p23104_i2': 'bmi_at_imaging',
    'p1160_i0': 'sleeping_duration_at_assessment',
    'p1160_i2': 'sleeping_duration_at_imaging',
    'p20160_i0': 'ever_smoked_at_assessment',
    'p20160_i2': 'ever_smoked_at_imaging',
    'p40007_i0': 'age_at_death',
    'p53_i0': 'date_of_assessment',
    'p53_i1': 'date_of_repeat_assessment',
    'p53_i2': 'date_of_imaging',
    'p53_i3': 'date_of_repeat_imaging',
},
               inplace = True )

# year of birth: p34
# month of birth: p52
merged[ 'date_of_birth' ] = pd.to_datetime(
    dict(
        year = merged[ 'p34' ],
        month = merged[ 'p52' ],
        day = 15                            # assuming mid-month for missing day information
    ),
    errors = 'coerce' )

# show histogram of date_of_birth
merged[ 'date_of_birth' ].hist( bins = 50 )

# convert date_of_assessment, date_of_repeat_assessment, date_of_imaging, date_of_repeat_imaging to datetime
merged[ 'date_of_assessment' ] = pd.to_datetime( merged[ 'date_of_assessment' ], errors = 'coerce' )
merged[ 'date_of_repeat_assessment' ] = pd.to_datetime( merged[ 'date_of_repeat_assessment' ], errors = 'coerce' )
merged[ 'date_of_imaging' ] = pd.to_datetime( merged[ 'date_of_imaging' ], errors = 'coerce' )
merged[ 'date_of_repeat_imaging' ] = pd.to_datetime( merged[ 'date_of_repeat_imaging' ], errors = 'coerce' )

# UKB p21003 is whole years; onset ages are continuous. Recompute baseline age
# from dates so both are on the same scale.
age_int = merged[ 'age_at_assessment' ].copy()   # original integer age, kept for the check
merged[ 'age_at_assessment' ] = ( merged[ 'date_of_assessment' ] - merged[ 'date_of_birth' ] ).dt.days / 365.25

diff = merged[ 'age_at_assessment' ] - age_int
print( diff.describe() )
print( "NaN baseline ages:", merged[ 'age_at_assessment' ].isna().sum() )

# compute age at last follow-up using date_of_birth and the latest of date_of_assessment, date_of_repeat_assessment, date_of_imaging, date_of_repeat_imaging, age_at_death
merged[ 'last_followup_date' ] = merged[ [ 'date_of_assessment', 'date_of_repeat_assessment', 'date_of_imaging', 'date_of_repeat_imaging' ] ].max( axis = 1 )
merged[ 'age_at_last_followup' ] = ( merged[ 'last_followup_date' ] - merged[ 'date_of_birth' ] ).dt.days / 365.25
# if age_at_death is available, use it
merged.loc[ ~merged[ 'age_at_death' ].isna(), 'age_at_last_followup' ] = merged.loc[ ~merged[ 'age_at_death' ].isna(), 'age_at_death' ]

merged = recode_variable( merged, codings_df, data_dict_df, 'p1558_i0', replace_name = 'alcohol_intake_at_assessment' )
merged = recode_variable( merged, codings_df, data_dict_df, 'p1558_i2', replace_name = 'alcohol_intake_at_imaging' )
merged = recode_variable( merged, codings_df, data_dict_df, 'p2110_i0', replace_name = 'able_to_confide_at_assessment' )
merged = recode_variable( merged, codings_df, data_dict_df, 'p2110_i2', replace_name = 'able_to_confide_at_imaging' )
merged = recode_variable( merged, codings_df, data_dict_df, 'p6160_i0', replace_name = 'leisure_social_activity_at_assessment' )
merged = recode_variable( merged, codings_df, data_dict_df, 'p6160_i2', replace_name = 'leisure_social_activity_at_imaging' )

# NaN values should remain NaN after recoding
merged[ 'genotyping_array' ] = [ 0 if x < 0 else 1 for x in merged[ 'p22000' ] ]
merged.loc[ merged[ 'p22000' ].isna(), 'genotyping_array' ] = np.nan

merged[ 'age_diff_assessment_imaging' ] = merged[ 'age_at_imaging' ] - merged[ 'age_at_assessment' ]
# merged['age_diff_assessment_imaging'].hist(bins=50)

# Rename principal component columns
merged.rename( columns = { f"p22009_a{i+1}": f"PC_{i+1}" for i in range( 40 ) }, inplace = True )

# in sleeping_duration_at_assessment and sleeping_duration_at_imaging, replace negative values with NaN
merged.loc[ merged[ 'sleeping_duration_at_assessment' ] < 0, 'sleeping_duration_at_assessment' ] = np.nan
merged.loc[ merged[ 'sleeping_duration_at_imaging' ] < 0, 'sleeping_duration_at_imaging' ] = np.nan

In [ ]:
relevant_columns_general = [
    'age_at_assessment',
    'age_at_imaging',
    'age_diff_assessment_imaging',
    'age_at_death',
    'date_of_assessment',
    'date_of_repeat_assessment',
    'date_of_imaging',
    'date_of_repeat_imaging',
    'date_of_birth',
    'age_at_last_followup',
    'sex',
    'townsend_deprivation_index',
    'body_fat_percentage_at_assessment',
    'body_fat_percentage_at_imaging',
    'bmi_at_assessment',
    'bmi_at_imaging',
    'sleeping_duration_at_assessment',
    'sleeping_duration_at_imaging',
    'ever_smoked_at_assessment',
    'ever_smoked_at_imaging']

relevant_columns_general += columns_with_prefix( merged, 'alcohol_intake_' )
relevant_columns_general += columns_with_prefix( merged, 'able_to_confide_' )
relevant_columns_general += columns_with_prefix( merged, 'leisure_social_activity_' )

relevant_columns_general += [ 'genotyping_array' ]
relevant_columns_general += [ f"PC_{i+1}" for i in range( 20 ) ]

In [ ]:
# Now adjust the direction of cognitive factors by making higher scores better

# Iterate through the rows in showcase['CF']
# Whenever the 'Field ID' matches one of the columns in merged (except 'eid'), where the match is defined as the column name starting with 'p' + Field ID + '_', and
# the 'Direction' is 'Negative', multiply the corresponding column in merged by -1 to make higher scores better
cf_showcase = showcase['CF']
for index, row in cf_showcase.iterrows():
    field_id = row['Field ID']
    direction = row['Direction']
    if direction == 'Negative':
        for col in merged.columns:
            if col.startswith(f'p{field_id}_'):
                print(f"Multiplying column {col} by -1, as Field ID {field_id} has Negative direction")
                merged[col] = merged[col] * -1

In [ ]:
# 6348, 6350, 6372, 20139, 20197, 21004, 23324, 26302 were only measured at the first imaging visit

CF_at_assessment = ['p399_i0_a2','p4282_i0','p20016_i0','p20023_i0']
CF_at_imaging = ['p399_i2_a2','p4282_i2','p6348_i2','p6350_i2','p6373_i2','p20016_i2','p20023_i2','p20139_i2','p20197_i2','p21004_i2','p23324_i2','p26302_i2']

relevant_columns_CF = CF_at_assessment + CF_at_imaging

In [ ]:
# Now, create two compound variables, one for childhood adversity and one for adulthood adversity
adversitie_showcase = showcase[ 'Adversities' ]
childhood_adversity_fields = []
adulthood_adversity_fields = []
for index, row in adversitie_showcase.iterrows():
    field_id = row[ 'Field ID' ]
    timing = row[ 'Group' ]
    if timing == 'Childhood':
        new_id = f'p{field_id}_binary'
        merged[ new_id ] = merged[ f"p{field_id}" ].replace( { -3: np.nan, 0: 0, 1: 1, 2: 1, 3: 1, 4: 1 } )
        if row[ 'Direction' ] == 'Positive':
            merged[ new_id ] = merged[ new_id ].replace( { 1: 0, 0: 1 } )
        childhood_adversity_fields.append( new_id )
    elif timing == 'Adulthood':
        new_id = f'p{field_id}_binary'
        merged[ new_id ] = merged[ f"p{field_id}" ].replace( { -3: np.nan, 0: 0, 1: 1, 2: 1 } )
        if row[ 'Direction' ] == 'Positive':
            merged[ new_id ] = merged[ new_id ].replace( { 1: 0, 0: 1 } )
        adulthood_adversity_fields.append( new_id )

merged[ 'childhood_adversity_score' ] = merged[ childhood_adversity_fields ].sum( axis = 1, skipna = True )
merged.loc[ merged[ childhood_adversity_fields ].isna().all( axis = 1 ), 'childhood_adversity_score' ] = np.nan
merged[ 'adulthood_adversity_score' ] = merged[ adulthood_adversity_fields ].sum( axis = 1, skipna = True )
merged.loc[ merged[ adulthood_adversity_fields ].isna().all( axis = 1 ), 'adulthood_adversity_score' ] = np.nan

merged[ 'childhood_adversity_binary' ] = np.where( merged[ 'childhood_adversity_score' ] > 0, 1, 0 )
merged.loc[ merged[ 'childhood_adversity_score' ].isna(), 'childhood_adversity_binary' ] = np.nan
merged[ 'adulthood_adversity_binary' ] = np.where( merged[ 'adulthood_adversity_score' ] > 0, 1, 0 )
merged.loc[ merged[ 'adulthood_adversity_score' ].isna(), 'adulthood_adversity_binary' ] = np.nan

relevant_columns_adversities = [ 'childhood_adversity_score', 'childhood_adversity_binary', 'adulthood_adversity_score', 'adulthood_adversity_binary' ]

In [ ]:
print( merged[ 'childhood_adversity_score' ].value_counts( dropna = False ) )
print( merged[ 'adulthood_adversity_score' ].value_counts( dropna = False ) )
print( merged[ 'childhood_adversity_binary' ].value_counts( dropna = False ) )
print( merged[ 'adulthood_adversity_binary' ].value_counts( dropna = False ) )

In [ ]:
relevant_columns_NMR = [
    'p23444_i0',         #	Omega-3 Fatty Acids
    'p23451_i0',         # Omega-3 Fatty Acids to Total Fatty Acids percentage
    'p23445_i0',         # Omega-6 Fatty Acids
    'p23459_i0',         # Omega-6 Fatty Acids to Omega-3 Fatty Acids ratio
    'p23452_i0',        #	Omega-6 Fatty Acids to Total Fatty Acids percentage
]

# # Visualize the distributions of the NMR relevant columns
# import matplotlib.pyplot as plt
# import seaborn as sns   
# for col in NMR_relevant_columns:
#     plt.figure(figsize=(8, 4))
#     sns.histplot(merged[col].dropna(), bins=50, kde=True)
#     plt.title(f'Distribution of {col}')
#     plt.xlabel(col)
#     plt.ylabel('Frequency')
#     plt.show()

In [ ]:
import warnings
from pandas.errors import PerformanceWarning

with warnings.catch_warnings():
    warnings.simplefilter("ignore", PerformanceWarning)

    include_diseases_onset_dates = False
    
    # from the showcase, iterate through the 'Diseases' sheet to get relevant disease columns
    # then compute the onset ages for each disease, use the ICD10 codes to identify cases as new columns
    # the corresponding field id column in merged contains the first date of diagnosis
    # for each disease, create a new column by the ICD10 code, and set the value to the age at diagnosis if the person has that disease
    # where age is computed as the difference between date of first diagnosis and date of birth
    # There are some special dates, which should be treated as missing values:
    # 1900-01-01	Code has no event date
    # 1901-01-01	Code has event date before participant's date of birth
    # 1902-02-02	Code has event date matching participant's date of birth
    # 1903-03-03	Code has event date after participant's date of birth and falls in the same calendar year as date of birth
    # 1909-09-09	Code has event date in the future and is presumed to be a place-holder or other system default
    # 2037-07-07	Code has event date in the future and is presumed to be a place-holder or other system default
    special_dates = [ '1900-01-01', '1901-01-01', '1902-02-02', '1903-03-03', '1909-09-09', '2037-07-07' ]
    relevant_columns_diseases = []
    relevant_columns_diseases_dates = []
    
    for i in range( 4 ):
        disease_showcase = showcase[ f'Diseases_{i+1}' ]
    
        for index, row in disease_showcase.iterrows():
            field_id = f"p{row['Field ID']}"
            icd10_code = str( row[ 'Description' ] ).strip()
    
            if field_id in merged.columns:
                # merged[ icd10_code ] = np.nan
                # merged[ icd10_code + '_date' ] = np.nan
                # Convert the diagnosis date to datetime
                diagnosis_dates = pd.to_datetime( merged[ field_id ], errors = 'coerce' )
                # Set special dates to NaT
                diagnosis_dates = diagnosis_dates.replace( pd.to_datetime( special_dates ), pd.NaT )
                # Calculate age at diagnosis
                age_at_diagnosis = ( diagnosis_dates - merged[ 'date_of_birth' ] ).dt.days / 365.25
                # Assign age at diagnosis to the new column where diagnosis date is not NaT
                merged.loc[ diagnosis_dates.notna(), icd10_code ] = age_at_diagnosis[ diagnosis_dates.notna() ]
                relevant_columns_diseases.append( icd10_code )            
                # Also assign the diagnosis date to the corresponding date column
                if include_diseases_onset_dates:
                    merged.loc[ diagnosis_dates.notna(), icd10_code + '_date' ] = diagnosis_dates[ diagnosis_dates.notna() ]
                    relevant_columns_diseases_dates.append( icd10_code + '_date' )
           
    # algorithmically defined outcomes in Other_Diseases         
    disease_showcase = showcase[ "Other_Diseases" ]
    
    for index, row in disease_showcase.iterrows():
        field_id = f"p{row['Field ID']}"
        dname = str( row[ 'Name' ] ).strip()
    
        if field_id in merged.columns and dname != 'nan' and dname != '':
            # Convert the diagnosis date to datetime
            diagnosis_dates = pd.to_datetime( merged[ field_id ], errors = 'coerce' )
            # Set special dates to NaT
            diagnosis_dates = diagnosis_dates.replace( pd.to_datetime( special_dates ), pd.NaT )
            # Calculate age at diagnosis
            age_at_diagnosis = ( diagnosis_dates - merged[ 'date_of_birth' ] ).dt.days / 365.25
            # Assign age at diagnosis to the new column where diagnosis date is not NaT
            merged.loc[ diagnosis_dates.notna(), dname ] = age_at_diagnosis[ diagnosis_dates.notna() ]
            relevant_columns_diseases.append( dname )            
            # Also assign the diagnosis date to the corresponding date column
            if include_diseases_onset_dates:
                merged.loc[ diagnosis_dates.notna(), dname + '_date' ] = diagnosis_dates[ diagnosis_dates.notna() ]
                relevant_columns_diseases_dates.append( dname + '_date' )


In [ ]:
# count negative values in the disease onset age columns
all_negative_counts = 0
for col in relevant_columns_diseases:
    negative_count = ( merged[ col ] <= 0 ).sum()
    all_negative_counts += negative_count

print( f"Total negative values across all disease onset age columns: {all_negative_counts}" )

# clear the negative values by setting them to NaN
for col in relevant_columns_diseases:
    merged.loc[ merged[ col ] <= 0, col ] = np.nan
    if include_diseases_onset_dates:
        merged.loc[ merged[ col ] <= 0, f"{col}_date" ] = np.nan

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", PerformanceWarning)
    
    # the parental illnesses variables are an increasing set of illnesses reported for mother and father
    # first, we compute the union of all reported illnesses for mother and father across the four visits for each row in the merged dataframe, separately for mother and father
    # father columns: 'p20107_i0', 'p20107_i1', 'p20107_i2', 'p20107_i3'
    # mother columns: 'p20110_i0', 'p20110_i1', 'p20110_i2', 'p20110_i3'
    # father_columns = [ 'p20107_i0', 'p20107_i1', 'p20107_i2', 'p20107_i3' ]
    # mother_columns = [ 'p20110_i0', 'p20110_i1', 'p20110_i2', 'p20110_i3' ]
    
    # Use baseline-visit parental illness only
    father_columns = [ 'p20107_i0' ]
    mother_columns = [ 'p20110_i0' ]
    
    parental_diseases = [ {
        'name': 'alzheimer',
        'code': '10',
        'group': '1'
    }, {
        'name': 'highbloodpressure',
        'code': '8',
        'group': '1'
    }, {
        'name': 'severedepression',
        'code': '12',
        'group': '2'
    }, {
        'name': 'parkinson',
        'code': '11',
        'group': '2'
    } ]
    
    father_columns_by_diseases = {}
    for c in father_columns:
        illnesses_column = f"{c}_illnesses"
        merged[ illnesses_column ] = merged[ c ].apply( lambda x: set( map( str.strip,
                                                                            str( x ).replace( '[', '' ).replace( ']', '' ).split( ',' ) ) )
                                                        if pd.notna( x ) else set() )
        for disease in parental_diseases:
            name = f"{c}_{disease['name']}"
            if disease['group'] == '1':
                merged[ name ] = merged[ illnesses_column ].apply(
                    lambda illnesses: 0 if '-17' in illnesses else ( np.nan if ( '-11' in illnesses or '-13' in illnesses ) else ( 1 if disease['code'] in illnesses else 0 ) ) )
            elif disease['group'] == '2':
                merged[ name ] = merged[ illnesses_column ].apply(
                    lambda illnesses: 0 if '-27' in illnesses else ( np.nan if ( '-21' in illnesses or '-23' in illnesses ) else ( 1 if disease['code'] in illnesses else 0 ) ) )
                
            father_columns_by_diseases.setdefault( disease['name'], [] ).append( name )
            
    mother_columns_by_diseases = {}
    for c in mother_columns:
        illnesses_column = f"{c}_illnesses"
        merged[ illnesses_column ] = merged[ c ].apply( lambda x: set( map( str.strip,
                                                                            str( x ).replace( '[', '' ).replace( ']', '' ).split( ',' ) ) )
                                                        if pd.notna( x ) else set() )
        for disease in parental_diseases:
            name = f"{c}_{disease['name']}"
            if disease['group'] == '1':
                merged[ name ] = merged[ illnesses_column ].apply(
                    lambda illnesses: 0 if '-17' in illnesses else ( np.nan if ( '-11' in illnesses or '-13' in illnesses ) else ( 1 if disease['code'] in illnesses else 0 ) ) )
            elif disease['group'] == '2':
                merged[ name ] = merged[ illnesses_column ].apply(
                    lambda illnesses: 0 if '-27' in illnesses else ( np.nan if ( '-21' in illnesses or '-23' in illnesses ) else ( 1 if disease['code'] in illnesses else 0 ) ) )
                
            mother_columns_by_diseases.setdefault( disease['name'], [] ).append( name )
    
    
    for disease in parental_diseases:
        name = disease['name']
        father_cols = father_columns_by_diseases[ name ]
        mother_cols = mother_columns_by_diseases[ name ]
        
        father_combined_col = f"father_{name}"
        mother_combined_col = f"mother_{name}"
        
        # Combine father columns using max to indicate presence if any visit indicates presence
        merged[ father_combined_col ] = merged[ father_cols ].max( axis = 1, skipna = True )
        # merged.loc[ merged[ father_cols ].isna().all( axis = 1 ), father_combined_col ] = np.nan
        
        # Combine mother columns using max to indicate presence if any visit indicates presence
        merged[ mother_combined_col ] = merged[ mother_cols ].max( axis = 1, skipna = True )
        # merged.loc[ merged[ mother_cols ].isna().all( axis = 1 ), mother_combined_col ] = np.nan
    
    # # then, for illness code '10' (Alzheimer) for parental illnesses, we create two new binary columns indicating presence or absence of that illness in father and mother
    # # if '-17' is present, it means 'None of the above', so we set the illness presence to 0 in that case
    # # if '-11' or '-13' is present, it means 'Do not know' or 'Prefer not to answer', so we set the illness presence to NaN in that case
    # merged[ 'father_alzheimers' ] = merged[ 'father_illnesses' ].apply(
    #     lambda illnesses: 0 if '-17' in illnesses else ( np.nan if ( '-11' in illnesses or '-13' in illnesses ) else ( 1 if '10' in illnesses else 0 ) ) )
    # merged[ 'mother_alzheimers' ] = merged[ 'mother_illnesses' ].apply(
    #     lambda illnesses: 0 if '-17' in illnesses else ( np.nan if ( '-11' in illnesses or '-13' in illnesses ) else ( 1 if '10' in illnesses else 0 ) ) )
    
    # merged[ 'father_highbloodpressure' ] = merged[ 'father_illnesses' ].apply(
    #     lambda illnesses: 0 if '-17' in illnesses else ( np.nan if ( '-11' in illnesses or '-13' in illnesses ) else ( 1 if '8' in illnesses else 0 ) ) )
    # merged[ 'mother_highbloodpressure' ] = merged[ 'mother_illnesses' ].apply(
    #     lambda illnesses: 0 if '-17' in illnesses else ( np.nan if ( '-11' in illnesses or '-13' in illnesses ) else ( 1 if '8' in illnesses else 0 ) ) )
    
    # merged[ 'father_severedepression' ] = merged[ 'father_illnesses' ].apply(
    #     lambda illnesses: 0 if '-27' in illnesses else ( np.nan if ( '-21' in illnesses or '-23' in illnesses ) else ( 1 if '12' in illnesses else 0 ) ) )
    # merged[ 'mother_severedepression' ] = merged[ 'mother_illnesses' ].apply(
    #     lambda illnesses: 0 if '-27' in illnesses else ( np.nan if ( '-21' in illnesses or '-23' in illnesses ) else ( 1 if '12' in illnesses else 0 ) ) )
    
    # merged[ 'father_parkinson' ] = merged[ 'father_illnesses' ].apply(
    #     lambda illnesses: 0 if '-27' in illnesses else ( np.nan if ( '-21' in illnesses or '-23' in illnesses ) else ( 1 if '11' in illnesses else 0 ) ) )
    # merged[ 'mother_parkinson' ] = merged[ 'mother_illnesses' ].apply(
    #     lambda illnesses: 0 if '-27' in illnesses else ( np.nan if ( '-21' in illnesses or '-23' in illnesses ) else ( 1 if '11' in illnesses else 0 ) ) )
    
    relevant_columns_parental_illnesses = [ f"father_{disease['name']}" for disease in parental_diseases ] + [ f"mother_{disease['name']}" for disease in parental_diseases ]
    
    for col in relevant_columns_parental_illnesses:
        print( f"{col} value counts:\n{merged[ col ].value_counts( dropna = False )}\n" )


## Quality filtering of GWAS

In [ ]:
merged = merged[ ( merged[ 'sex' ] == merged[ 'p22001' ] ) & # Filter in sex and genetic sex are the same
                  ( merged[ 'p22006' ] == 1 ) &               # in_white_british_ancestry_subset
                  ( merged[ 'p22019' ].isnull() ) &           # Not Sex chromosome aneuploidy
                  ( merged[ 'p22021' ] != 10 ) &               # Not Ten or more third-degree relatives identified (not 'excess_relatives')
                  ( merged[ 'p22027' ].isnull() )             # Not het_missing_outliers
                 ]

In [ ]:
# remove rows where all columns in relevant_columns_CF are NaN
# merged = merged[ ~merged[ relevant_columns_CF ].isna().all( axis = 1 ) ]

print( f"QCed shape: {merged.shape}" )

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", PerformanceWarning)
    
    merged[ 'IID' ] = merged[ 'eid' ]
    merged[ 'FID' ] = merged[ 'eid' ]
    
    relevant_columns = [ 'FID', 'IID' ] + relevant_columns_general + relevant_columns_CF + relevant_columns_adversities + relevant_columns_NMR + relevant_columns_diseases + relevant_columns_diseases_dates + relevant_columns_parental_illnesses
    
    df_phenotypes = merged[ relevant_columns ].copy()

In [ ]:
print( df_phenotypes.shape )   # do not display participant rows: notebook outputs must stay free of individual-level data

In [ ]:
# crosstab between (father_severedepression or mother_severedepression) and childhood_adversity_binary
pd.crosstab( index = ( df_phenotypes[ 'father_severedepression' ] == 1 ) | ( df_phenotypes[ 'mother_severedepression' ] == 1 ),
             columns = df_phenotypes[ 'childhood_adversity_binary' ] )

# fishers exact test on the crosstab
from scipy.stats import fisher_exact
contingency_table = pd.crosstab( index = ( df_phenotypes[ 'mother_severedepression' ] == 1 )  ,
                                 columns = df_phenotypes[ 'childhood_adversity_binary' ] )
oddsratio, p_value = fisher_exact( contingency_table )
print( f"Fisher's exact test p-value: {p_value}, odds ratio: {oddsratio}" )

In [ ]:
# crosstab between 'G30' binary status and 'ado_alzheimer' binary status, both indicating presence of Alzheimer's disease converted from age of onset columns
pd.crosstab( df_phenotypes[ 'G30' ].notna(), df_phenotypes[ 'ado_alzheimer' ].notna(), dropna = False )



In [ ]:
relevant_columns_general_at_assessment = [ col for col in relevant_columns_general if col.find( '_at_assessment' ) != -1 ]

# remove rows where any columns in relevant_columns_general_at_assessment is NaN
# remove rows where any columns in relevant_columns_NMR is NaN
# df_phenotypes = df_phenotypes[ ~df_phenotypes[ relevant_columns_general_at_assessment ].isna().any( axis = 1 ) ]
df_phenotypes = df_phenotypes[ ~df_phenotypes[ relevant_columns_NMR ].isna().any( axis = 1 ) ]
df_phenotypes = df_phenotypes[ ~df_phenotypes[ 'townsend_deprivation_index' ].isna() ]
print( f"QCed shape: {df_phenotypes.shape}" )

In [ ]:
relevant_columns_general_at_assessment

In [ ]:
df_phenotypes[ relevant_columns_general_at_assessment].isna().sum( axis = 1).value_counts()

In [ ]:
df_phenotypes = df_phenotypes.sort_values( by = 'IID' ).reset_index( drop = True )

In [ ]:
# Write phenotype files to a TSV file
preprocessed_dir = PREPROCESSED_DIR   # default /preprocessed/UKB (outside the repository), override with UKB_PREPROCESSED_DIR
os.makedirs(preprocessed_dir, exist_ok=True)

df_phenotypes.to_csv( f'{preprocessed_dir}/phenotypes.tsv', sep = '\t', na_rep = 'NA', index = False, quoting = 3 )
print( "Shape of the final phenotype dataframe:", df_phenotypes.shape )

# Exploratory checks (not used in the manuscript)
Everything below runs after `phenotypes.tsv` has been written and does not change it. The cognitive-score residuals and plots were exploratory and are not used by the experiments.

In [ ]:
df_phenotypes['G30_binary'] = np.where( np.isnan( df_phenotypes['G30']), 0, 1 )
print( df_phenotypes['G30_binary'].value_counts() )

df_phenotypes['Dementia'] = np.where( ( ~np.isnan( df_phenotypes['G30'] ) ) | ( ~np.isnan( df_phenotypes['F00'] ) ) | ( ~np.isnan( df_phenotypes['F01'] ) ) | ( ~np.isnan( df_phenotypes['F02'] ) ) | ( ~np.isnan( df_phenotypes['F03'] ) ), 1, 0 )
print( df_phenotypes['Dementia'].value_counts() )

df_phenotypes['F32_binary'] = np.where( np.isnan( df_phenotypes['F32']), 0, 1 )
print( df_phenotypes['F32_binary'].value_counts() )

In [ ]:
# Write phenotype files to a TSV file
# df_phenotypes.to_csv('phenotypes_with_diseases.tsv', sep='\t', na_rep='NA', index=False, quoting=3)

## Visualizations

In [ ]:
cf_initial_assessment = merged[CF_at_assessment].dropna(axis=0,how='all',subset=[i for i in CF_at_assessment if i != 'eid'])
cf_initial_imaging = merged[CF_at_imaging].dropna(axis=0,how='all',subset=[i for i in CF_at_imaging if i != 'eid'])

In [ ]:
print(cf_initial_assessment.shape)
print(cf_initial_imaging.shape)

In [ ]:
cf_initial_assessment.describe()

In [ ]:
# compute pairwise correlations between the cognitive scores at initial assessment, and plot as a heatmap
import seaborn as sns
import matplotlib.pyplot as plt
corr = cf_initial_assessment.drop(columns=['eid'], errors='ignore').corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Pairwise Correlations between Cognitive Scores at Initial Assessment")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
corr = cf_initial_imaging.drop(columns=['eid'], errors='ignore').corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Pairwise Correlations between Cognitive Scores at Imaging Visit")
plt.show()

In [ ]:
# compute pairwise correlations between the cognitive scores at imaging visit, and plot as a heatmap
# also show dendrogram to cluster the cognitive scores
import seaborn as sns
import matplotlib.pyplot as plt
corr = cf_initial_imaging.drop(columns=['eid'], errors='ignore').corr()
plt.figure(figsize=(10,8))
sns.clustermap(corr, annot=True, fmt=".2f", cmap='coolwarm')
plt.title("Pairwise Correlations between Cognitive Scores at Imaging Visit")
plt.show()


In [ ]:
# show pairwise plots between 'p26302_i2', 'p4282_i2', 'p20139_i2'
import seaborn as sns
import matplotlib.pyplot as plt
pairwise_data = cf_initial_imaging[['p26302_i2', 'p20016_i2']]
sns.pairplot(pairwise_data)
plt.suptitle("Pairwise Plots between p26302_i2, p20016_i2", y=1.02)
plt.show()

## Computing residuals and z-scores

In [ ]:
[
    "age_at_assessment", "sex", "townsend_deprivation_index", "body_fat_percentage_at_assessment", "bmi_at_assessment", "sleeping_duration_at_assessment",
    "p1031_i0"
] + columns_with_prefix( merged, "alcohol_intake_at_assessment" ) + \
                                               columns_with_prefix( merged, "able_to_confide_at_assessment" ) + \
                                               columns_with_prefix( merged, "leisure_social_activity_at_assessment" )

In [ ]:
# We first compute the residuals of each cognitive score with respect to age, sex, etc.
import statsmodels.api as sm
import statsmodels.formula.api as smf


def compute_residuals_predict( df, score_col, predictors, verbose = False ):
    cols = [ score_col ] + predictors
    fit_df = df[ cols ].dropna()

    formula = f"{score_col} ~ " + " + ".join( predictors )
    print( f"Fitting model: {formula} on {fit_df.shape[0]} rows" )
    model = smf.ols( formula, data = fit_df ).fit()

    print( model.summary() )

    # predict for everyone who has predictors
    pred = model.predict( df[ predictors ] )
    residuals = df[ score_col ] - pred

    # keep NaN where score is missing
    residuals[ df[ score_col ].isna() ] = np.nan

    if verbose:
        # show a histogram of the residuals
        residuals.hist( bins = 50 )
        plt.title( f"Histogram of Residuals for {col} at Initial Assessment" )
        plt.show()

        # correlation with age (only rows used)
        ok = residuals.notna() & merged[ "age_at_assessment" ].notna()
        corr = np.corrcoef( residuals.loc[ ok ], merged.loc[ ok, "age_at_assessment" ] )[ 0, 1 ]

        print( f"{col}: corr(residuals, age) = {corr:.3f}" )

        # show scatterplot of residuals vs original score
        plt.scatter( merged[ col ], residuals, alpha = 0.5 )
        plt.xlabel( f"Original Score: {col}" )
        plt.ylabel( "Residuals" )
        plt.title( f"Scatterplot of Residuals vs Original Score for {col} at Initial Assessment" )
        plt.show()

    return residuals


for col in cf_initial_assessment:
    if col != 'eid':
        residuals = compute_residuals_predict( merged,
                                               col,
                                               predictors = [
                                                   "age_at_assessment", "sex", "townsend_deprivation_index", "body_fat_percentage_at_assessment",
                                                   "bmi_at_assessment", "sleeping_duration_at_assessment", "p1031_i0"
                                               ] + columns_with_prefix( merged, "alcohol_intake_at_assessment" ) +
                                               columns_with_prefix( merged, "able_to_confide_at_assessment" ) +
                                               columns_with_prefix( merged, "leisure_social_activity_at_assessment" ),
                                               verbose = True )

        # add the residuals as a new column to merged
        # as the original score was positive, the residuals will also be positive for better-than-expected performance
        merged[ f"{col}_residuals" ] = residuals

for col in cf_initial_imaging:
    if col != 'eid':
        original_col = col.replace( '_i2', '_i0' )                                                                                                        # original score at assessment
        if original_col not in merged.columns:
            print( f"Warning: original column {original_col} not found in merged dataframe." )
            continue
        residuals = compute_residuals_predict( merged,
                                               col,
                                               predictors = [
                                                   original_col, "age_at_assessment", "age_diff_assessment_imaging", "sex", "townsend_deprivation_index",
                                                   "body_fat_percentage_at_imaging", "bmi_at_imaging", "sleeping_duration_at_imaging", "p1031_i0"
                                               ] + columns_with_prefix( merged, "alcohol_intake_at_imaging" ) +
                                               columns_with_prefix( merged, "able_to_confide_at_imaging" ) +
                                               columns_with_prefix( merged, "leisure_social_activity_at_imaging" ),
                                               verbose = True )
